# Fourth Down Decision Tool

**Goal:** For every 4th down, estimate the expected points of each option (go for it, kick a field goal, punt) and recommend the best one.

**How it works:** the three decision models predict what happens, and the expected points (EP) model puts a value on where each outcome leaves the game.

```
GO    = P(convert) x EP(us, 1st & 10 at ytg - distance)  +  (1 - P(convert)) x -EP(them, 1st & 10 at 100 - ytg)
FG    = P(make) x 3                                       +  (1 - P(make))    x -EP(them, 1st & 10 at min(100 - ytg, 80))
PUNT  = -EP(them, 1st & 10 at predicted opp_start_ytg)
```

The code lives in `src/cfb/models/fourth_down.py`. From a terminal, `python -m cfb.models.fourth_down --season 2025` scores a season and saves it to `marts.fourth_down_decisions`. This notebook uses the same functions to look at the results.

In [ ]:
from cfb.ml import *
from cfb.models import fourth_down as fd

models = fd.load_models()                 # the four saved models
plays = fd.load_plays(2025)               # every regulation 4th down in 2025 (~20s)
results = fd.score(plays, models)         # one row per play: values, recommendation, EP lost
results.head()

## Sanity check: spots with an obvious answer

Same game situation (a real 1st-quarter play with a tied score), moved to different spots. The first five are the ones `tests/test_fourth_down.py` checks; the last two are close calls, shown for interest. If an obvious one comes out wrong, `p_convert`, `p_make` and `pred_opp_start` show which model is off.

In [ ]:
base = plays[(plays['period'] == 1) & (plays['score_margin'] == 0)].iloc[[0]]

spots = {name: (ytg, dist) for name, (ytg, dist, _) in fd.SANITY_SPOTS.items()}
spots.update({"4th & 6 at opp 15 (close)": (15, 6), "4th & 4 at opp 38 (close)": (38, 4)})

checks = fd.score(pd.concat([fd.scenario(base, ytg, dist) for ytg, dist in spots.values()],
                            ignore_index=True), models)
checks.index = spots.keys()
checks[['p_convert', 'p_make', 'pred_opp_start', 'ev_go', 'ev_fg', 'ev_punt', 'recommend', 'margin']].round(2)

## Every 4th down in the season

In [ ]:
fd.summary(results)    # agreement rate, EP lost per 4th down, coach vs model crosstab

### Clearest disagreements

Plays where the coach's choice cost the most expected points. Many are blowouts: EP values points, not wins, so a team down 30 going for it is often right even when EP says kick.

In [ ]:
results.sort_values('ep_lost', ascending=False).head(15)[
    ['week', 'offense', 'defense', 'period', 'distance', 'yards_to_goal', 'score_margin',
     'coach_decision', 'recommend', 'p_convert', 'p_make', 'ev_go', 'ev_fg', 'ev_punt', 'ep_lost']
].round(2)

## From the saved table

Once a season is scored from the terminal, the results are a table, so analysis is plain SQL. Teams that lose the most expected points per 4th down in competitive situations (within two scores):

In [ ]:
team_query = '''
SELECT
    offense,
    count(*)                                              AS fourth_downs,
    round(avg((coach_decision = recommend)::int), 3)      AS agree_rate,
    round(avg((coach_decision = 'go')::int), 3)           AS coach_go_rate,
    round(avg((recommend = 'go')::int), 3)                AS model_go_rate,
    round(avg(ep_lost)::numeric, 3)                       AS ep_lost_per_4th
FROM marts.fourth_down_decisions
WHERE season = 2025
  AND abs(score_margin) <= 16
GROUP BY offense
HAVING count(*) >= 40
ORDER BY ep_lost_per_4th DESC
'''
teams = pd.read_sql(team_query, engine)
teams.head(15)